# 01 - Data Understanding

## Objetivo

Esta etapa tem como objetivo compreender a estrutura do conjunto de dados antes do início das etapas de limpeza, análise exploratória e modelagem.

Serão investigadas características como:

- Dimensionalidade do dataset;
- Tipos das variáveis;
- Valores ausentes;
- Distribuições iniciais;
- Estatísticas descritivas;
- Possíveis inconsistências;
- Definição da variável alvo.

Ao final desta etapa será possível compreender a estrutura geral dos dados e identificar os primeiros desafios para a construção do modelo de Machine Learning:

- Quantas observações existem?

- Quantas variáveis existem?

- Quais são numéricas?

- Quais são categóricas?

- Existem valores ausentes?

- Existem variáveis redundantes?

- Existe alguma variável que não deverá entrar no modelo?

- Qual variável será utilizada como alvo?

In [3]:
from pathlib import Path
import pandas as pd

# Caminho do projeto
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Carregamento dos dados
csv_file = DATA_PATH / "data.csv"
df = pd.read_csv(csv_file)

In [10]:
# Dimensão do dataset
n_samples, n_features = df.shape

print("=" * 35)
print("DIMENSÃO DO DATASET")
print("=" * 35)
print(f"Número de observações : {n_samples}")
print(f"Número de atributos   : {n_features}")

DIMENSÃO DO DATASET
Número de observações : 11657
Número de atributos   : 8


O conjunto de dados possui 11.657 observações distribuídas em 8 atributos. Em uma primeira análise, o volume de dados sugere que há uma quantidade adequada de exemplos para a construção e avaliação de modelos supervisionados de regressão. Entretanto, a qualidade dos dados, a presença de valores ausentes e a representatividade das variáveis deverão ser avaliadas nas próximas etapas.

## Estrutura do conjunto de dados

Nesta etapa será analisada a estrutura geral do DataFrame, verificando os atributos disponíveis, seus respectivos tipos de dados e a existência de valores ausentes.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11657 entries, 0 to 11656
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   address   11657 non-null  str  
 1   district  11657 non-null  str  
 2   area      11657 non-null  int64
 3   bedrooms  11657 non-null  int64
 4   garage    11657 non-null  int64
 5   type      11657 non-null  str  
 6   rent      11657 non-null  int64
 7   total     11657 non-null  int64
dtypes: int64(5), str(3)
memory usage: 728.7 KB


A análise da estrutura do DataFrame indica que o conjunto de dados é composto por 11.657 observações distribuídas em oito atributos, não apresentando valores ausentes em nenhuma das variáveis. Essa característica simplifica as próximas etapas do projeto, reduzindo a necessidade de técnicas de imputação de dados.

Entre os atributos disponíveis, address, district e type são armazenados como texto (str) e representam informações categóricas relacionadas à localização e ao tipo do imóvel. Já area, bedrooms, garage, rent e total são variáveis numéricas do tipo inteiro (int64), descrevendo características estruturais do imóvel e valores monetários.

Além disso, o conjunto ocupa aproximadamente 729 KB de memória, indicando que poderá ser manipulado integralmente em memória durante as etapas de exploração e modelagem, sem limitações computacionais relevantes.

| Variável     | Tipo (Pandas) | Categoria         | Descrição                                                          | Unidade    |
| ------------ | ------------- | ----------------- | ------------------------------------------------------------------ | ---------- |
| **address**  | `str`         | Categórica        | Endereço do imóvel.                                                | —          |
| **district** | `str`         | Categórica        | Bairro onde o imóvel está localizado.                              | —          |
| **area**     | `int64`       | Numérica discreta | Área do imóvel.                                                    | m²         |
| **bedrooms** | `int64`       | Numérica discreta | Número de dormitórios do imóvel.                                   | quantidade |
| **garage**   | `int64`       | Numérica discreta | Número de vagas de garagem disponíveis.                            | quantidade |
| **type**     | `str`         | Categórica        | Tipo do imóvel (apartamento, casa, sobrado etc.).                  | —          |
| **rent**     | `int64`       | Numérica discreta | Valor mensal do aluguel.                                           | R$         |
| **total**    | `int64`       | Numérica discreta | Custo total do imóvel, incluindo aluguel, impostos e demais taxas. | R$         |


## Hipóteses Iniciais

Com base na documentação oficial do conjunto de dados e na compreensão inicial das variáveis, algumas hipóteses podem ser levantadas antes da realização da Análise Exploratória de Dados (EDA). Essas hipóteses servirão como ponto de partida para orientar as próximas etapas do projeto e deverão ser confirmadas ou refutadas por meio de análises estatísticas e visualizações.

### Hipótese 1 — A localização influencia significativamente o valor do aluguel

A variável **`district`** representa o bairro onde o imóvel está localizado e, provavelmente, será uma das variáveis mais importantes para explicar a variação dos preços dos aluguéis. Espera-se que bairros com maior valorização imobiliária apresentem aluguéis médios superiores aos bairros periféricos.

---

### Hipótese 2 — Imóveis maiores tendem a possuir aluguéis mais elevados

A variável **`area`**, que representa a área construída do imóvel em metros quadrados, provavelmente possui uma relação positiva com o valor do aluguel. Espera-se que imóveis com maior área apresentem, em média, preços mais elevados.

---

### Hipótese 3 — O número de dormitórios está associado ao valor do aluguel

A variável **`bedrooms`** pode exercer influência sobre o preço do imóvel, uma vez que imóveis com maior quantidade de dormitórios costumam atender famílias maiores e, consequentemente, apresentar maior valor de locação. Entretanto, essa variável pode estar **correlacionada** com a área do imóvel, hipótese que será investigada posteriormente.

---

### Hipótese 4 — O número de vagas de garagem pode influenciar o preço

A variável **`garage`** representa a quantidade de vagas de garagem disponíveis. Espera-se que imóveis com maior número de vagas apresentem valores de aluguel superiores, especialmente em regiões de maior densidade urbana.

---

### Hipótese 5 — O tipo do imóvel influencia a distribuição dos preços

A variável **`type`** identifica o tipo do imóvel (apartamento, casa, sobrado, entre outros). É esperado que diferentes categorias apresentem distribuições distintas de valores de aluguel, refletindo características estruturais e de mercado específicas de cada tipo de imóvel.

---

### Hipótese 6 — A variável `address` pode apresentar alta cardinalidade

A variável **`address`** corresponde ao endereço do imóvel e provavelmente possui um grande número de valores distintos. Embora seja uma informação relevante para identificação do imóvel, sua utilização direta em modelos de Machine Learning pode não ser adequada, exigindo técnicas específicas de engenharia de atributos ou até mesmo sua remoção durante o pré-processamento.

---

### Hipótese 7 — A variável `total` pode causar vazamento de informação (*Data Leakage*)

A variável **`total`** representa o custo total do imóvel, incluindo aluguel, impostos e demais taxas. Como esse atributo é composto, em parte, pelo próprio valor do aluguel (`rent`), existe a possibilidade de que sua utilização como variável preditora introduza vazamento de informação (*Data Leakage*), comprometendo a avaliação do modelo. Essa hipótese será investigada durante a Análise Exploratória de Dados e considerada na etapa de seleção de atributos.

---

## Considerações

As hipóteses apresentadas nesta seção foram elaboradas antes da realização da Análise Exploratória de Dados (EDA) e têm como objetivo orientar a investigação do conjunto de dados. Ao longo do projeto, cada hipótese será validada ou refutada com base em evidências estatísticas, visualizações e análises quantitativas.


# Primeiras Análises Estatísticas

In [14]:
df.describe()

,area,bedrooms,garage,rent,total
count,11657.000000,11657.000000,11657.000000,11657.000000,11657.000000
mean,84.655658,1.966286,1.060393,3250.814789,4080.030625
std,74.020536,0.931313,1.132349,2650.711557,3352.480274
min,0.000000,0.000000,0.000000,500.000000,509.000000
25%,40.000000,1.000000,0.000000,1590.000000,1996.000000
50%,60.000000,2.000000,1.000000,2415.000000,3057.000000
75%,96.000000,3.000000,2.000000,3800.000000,4774.000000
max,580.000000,6.000000,6.000000,25000.000000,28700.000000


## Interpretação das Estatísticas Descritivas

A análise das estatísticas descritivas revela que o conjunto de dados apresenta características típicas de problemas de regressão envolvendo preços de imóveis.

As variáveis area, bedrooms e garage possuem valores mínimos iguais a zero. Embora esses registros possam representar inconsistências ou erros de cadastro, será necessária uma investigação adicional para verificar se correspondem a observações válidas ou se deverão ser tratados durante a etapa de preparação dos dados.

Observa-se ainda que as variáveis area, rent e total apresentam médias superiores às respectivas medianas, sugerindo distribuições assimétricas à direita. Esse comportamento indica a existência de um pequeno número de imóveis significativamente maiores ou mais caros do que a maior parte das observações, influenciando a média do conjunto.

Os valores máximos de área (580 m²), aluguel (R$ 25.000) e custo total (R$ 28.700) reforçam essa hipótese e indicam a necessidade de investigar a presença de possíveis valores extremos (outliers) durante a Análise Exploratória de Dados.

Por fim, a análise dos quartis mostra que 50% dos imóveis possuem área entre 40 m² e 96 m², enquanto os valores de aluguel concentram-se entre aproximadamente R$ 1.590 e R$ 3.800, com mediana de R$ 2.415, sugerindo que o conjunto representa predominantemente imóveis de pequeno e médio porte.

## Distribuição das Variáveis Categóricas

Nesta etapa serão analisadas as variáveis categóricas do conjunto de dados, buscando compreender as categorias existentes, sua frequência de ocorrência e possíveis desbalanceamentos.

In [20]:
df['type'].value_counts()

type
Apartamento           7194
Casa                  2841
Studio e kitnet       1381
Casa em condomínio     241
Name: count, dtype: int64

A distribuição da variável type indica predominância de Apartamento no conjunto de dados, representando a maior parte das observações com 7194 amostras. Os imóveis do tipo Studio e Kitnet representam uma parcela menor do conjunto de dados quando comparados aos apartamentos e casas. Já a categoria Casa em condomínio apresenta baixa representatividade, podendo exigir atenção durante análises estatísticas específicas.

Uma observação importante é que o dataset faz distinção entre Casa e Casa em condomínio, possuindo 2841 e 241 amostras, respectivamente. Nota-se que a existência desta distinção levanta a hipótese de que Casa em condomínio poderá possuir valores maiores de aluguel.

In [24]:
df['district'].value_counts()

district
Bela Vista                           352
Vila Mariana                         232
Jardim Paulista                      220
Centro                               178
Pinheiros                            159
                                    ... 
Vila Sargento José de Paula            1
Jardim Vitoria Regia (zona Oeste)      1
Vila São Francisco (zona Sul)          1
Jardim Vitoria Regia                   1
Retiro Morumbi                         1
Name: count, Length: 1199, dtype: int64

A variável district apresenta 1.199 categorias distintas, caracterizando uma variável categórica de alta cardinalidade. Os cinco distritos mais frequentes concentram aproximadamente 10% das observações, sugerindo que os imóveis estão distribuídos por diversas regiões da cidade.

Além disso, observa-se a existência de categorias com apenas um único registro, indicando baixa representatividade para determinadas localidades. Essa característica poderá influenciar tanto a análise estatística quanto a etapa de modelagem, exigindo atenção durante o processo de codificação das variáveis categóricas e possível tratamento de categorias raras.

## Conclusões

O conjunto de dados apresenta boa qualidade inicial, sem valores ausentes e com estrutura consistente para análise. Foram identificadas variáveis numéricas e categóricas relevantes para o problema, além de possíveis desafios relacionados à alta cardinalidade da variável district, à presença de valores mínimos iguais a zero em algumas variáveis e ao potencial vazamento de informação da variável total. Essas observações orientarão as etapas de análise exploratória e preparação dos dados.